In [ ]:
# Install minimal dependencies
!pip install pycryptodome base58 tqdm --force-reinstall --no-cache-dir --quiet

import os
import time
import csv
import hashlib
import base58
from Crypto.Hash import RIPEMD160
from google.colab import files
import random
import logging
from tqdm import tqdm

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# Setup checkpoint directory
CHECKPOINT_DIR = '/content/ripemphantom_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Upload addresses.txt
logger.info("Upload addresses.txt with 1MVLP2kRPNqz8VJUy83LstUoMQzUjgq4Zg")
files.upload()
ADDRESS_FILE = 'addresses.txt'

# Constants
TARGET_ADDRESS = '1MVLP2kRPNqz8VJUy83LstUoMQzUjgq4Zg'
SEED_KEY = 0x4197113
HAMMING_THRESHOLD = 45
BEST_HAMMING = 39
INITIAL_RANGE = 10000000
MAX_KEY = 2**61
CHECKPOINT_INTERVAL = 10
MIN_HAMMING_FOR_MUTATION = 35
STAGNATION_THRESHOLD = 10
BATCH_SIZE = 10000

# Files
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, 'checkpoint.csv')
OUTPUT_CSV = os.path.join(CHECKPOINT_DIR, 'new_matches.csv')

# Helper functions
def privkey_to_pubkey(privkey):
    privkey_bytes = privkey.to_bytes(32, 'big')
    return hashlib.sha256(privkey_bytes).digest()

def pubkey_to_address(pubkey):
    sha256_hash = hashlib.sha256(pubkey).digest()
    ripemd160 = RIPEMD160.new()
    ripemd160.update(sha256_hash)
    hash160 = ripemd160.digest()
    versioned = b'\x00' + hash160
    checksum = hashlib.sha256(hashlib.sha256(versioned).digest()).digest()[:4]
    return base58.b58encode(versioned + checksum).decode()

def hamming_distance(hash1, hash2):
    return bin(int.from_bytes(hash1, 'big') ^ int.from_bytes(hash2, 'big')).count('1')

def hash160_from_address(address):
    try:
        decoded = base58.b58decode(address)
        return decoded[1:-4]
    except Exception as e:
        logger.error(f"Invalid address {address}: {e}")
        raise

# Hardcoded CSV
def read_csv(file_path):
    results = []
    if not os.path.exists(file_path):
        return results
    with open(file_path, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                results.append({
                    'Address': row['Address'],
                    'Private Key': row['Private Key'],
                    'Hamming Distance': float(row['Hamming Distance']),
                    'Similarity %': float(row['Similarity %']),
                    'Entropy Level': float(row['Entropy Level']),
                    'Exact Match': row['Exact Match'].lower() == 'true'
                })
            except:
                pass
    return results

def write_csv(file_path, results):
    with open(file_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['Address', 'Private Key', 'Hamming Distance', 'Similarity %', 'Entropy Level', 'Exact Match'])
        writer.writeheader()
        for result in results:
            writer.writerow(result)

# Key crafting: bit flips based on XOR
def craft_keys(base_key, target_hash160, current_hash160):
    xor = int.from_bytes(current_hash160, 'big') ^ int.from_bytes(target_hash160, 'big')
    bits = [int(b) for b in bin(xor)[2:].zfill(160)]
    flip_positions = [i for i, b in enumerate(bits) if b == 1][:20]  # Top 20 differing bits
    crafted_keys = []
    for pos in flip_positions:
        crafted_key = base_key ^ (1 << pos)
        if 1 <= crafted_key < MAX_KEY:
            crafted_keys.append(crafted_key)
    return crafted_keys

# Process batch
def process_batch(batch, target_hash160, min_hamming):
    results = []
    mutation_mode = min_hamming < MIN_HAMMING_FOR_MUTATION
    for key in batch:
        pubkey = privkey_to_pubkey(key)
        ripemd160 = RIPEMD160.new()
        ripemd160.update(hashlib.sha256(pubkey).digest())
        candidate_hash160 = ripemd160.digest()
        hamming = hamming_distance(candidate_hash160, target_hash160)
        if hamming <= HAMMING_THRESHOLD:
            address = pubkey_to_address(pubkey)
            similarity = (160 - hamming) / 160 * 100
            result = {
                'Address': address,
                'Private Key': hex(key),
                'Hamming Distance': hamming,
                'Similarity %': similarity,
                'Entropy Level': 0.0,
                'Exact Match': hamming == 0
            }
            results.append(result)
            if hamming < BEST_HAMMING:
                print(f"NEW BEST: Key {hex(key)}, Hamming {hamming}")
                logger.info(f"New best: Key {hex(key)}, Hamming {hamming}")
        if mutation_mode or hamming < BEST_HAMMING:
            crafted_keys = craft_keys(key, target_hash160, candidate_hash160)
            for crafted_key in crafted_keys:
                crafted_pubkey = privkey_to_pubkey(crafted_key)
                ripemd160 = RIPEMD160.new()
                ripemd160.update(hashlib.sha256(crafted_pubkey).digest())
                crafted_hash160 = ripemd160.digest()
                crafted_hamming = hamming_distance(crafted_hash160, target_hash160)
                if crafted_hamming <= HAMMING_THRESHOLD:
                    crafted_address = pubkey_to_address(crafted_pubkey)
                    crafted_similarity = (160 - crafted_hamming) / 160 * 100
                    result = {
                        'Address': crafted_address,
                        'Private Key': hex(crafted_key),
                        'Hamming Distance': crafted_hamming,
                        'Similarity %': crafted_similarity,
                        'Entropy Level': 0.0,
                        'Exact Match': crafted_hamming == 0
                    }
                    results.append(result)
                    if crafted_hamming < BEST_HAMMING:
                        print(f"NEW BEST (CRAFTED): Key {hex(crafted_key)}, Hamming {crafted_hamming}")
                        logger.info(f"Crafted key success: Key {hex(crafted_key)}, Hamming {crafted_hamming}")
    return results

# Load target address
try:
    with open(ADDRESS_FILE, 'r') as f:
        addresses = [line.strip() for line in f if line.strip()]
    if len(addresses) != 999 or TARGET_ADDRESS not in addresses:
        logger.error(f"Expected 999 addresses including {TARGET_ADDRESS}")
        raise ValueError("Invalid addresses.txt")
    target_hash160 = hash160_from_address(TARGET_ADDRESS)
except FileNotFoundError:
    logger.error(f"{ADDRESS_FILE} not found")
    raise
except Exception as e:
    logger.error(f"Error loading addresses: {e}")
    raise

# Initialize results
results = read_csv(CHECKPOINT_FILE)
if results:
    logger.info(f"Loaded {len(results)} results from checkpoint")

# Search loop
current_key = SEED_KEY
search_range = INITIAL_RANGE
iteration = 0
start_time = time.time()
last_checkpoint = start_time
min_hamming = BEST_HAMMING
stagnation_count = 0

while current_key < MAX_KEY:
    try:
        if stagnation_count >= STAGNATION_THRESHOLD:
            current_key = min(max(1, SEED_KEY + random.randint(-10**9, 10**9)), MAX_KEY - 1)
            search_range = INITIAL_RANGE
            stagnation_count = 0
            logger.info(f"Stagnation detected, jumping to key {hex(current_key)}")
        keys = list(range(max(1, current_key - search_range), current_key + search_range + 1))
        keys = [k for k in keys if 1 <= k < MAX_KEY]
        batches = [keys[i:i + BATCH_SIZE] for i in range(0, len(keys), BATCH_SIZE)]
        for batch in tqdm(batches, desc=f"Iteration {iteration}, Keys {current_key-search_range} to {current_key+search_range}"):
            batch_result = process_batch(batch, target_hash160, min_hamming)
            results.extend(batch_result)
            new_min_hamming = min([r['Hamming Distance'] for r in batch_result] + [min_hamming])
            if new_min_hamming < min_hamming:
                min_hamming = new_min_hamming
                stagnation_count = 0
                logger.info(f"Updated min Hamming: {min_hamming}")
            else:
                stagnation_count += 1
        current_time = time.time()
        if results and (current_time - last_checkpoint >= CHECKPOINT_INTERVAL):
            write_csv(OUTPUT_CSV, results)
            write_csv(CHECKPOINT_FILE, results)
            logger.info(f"Saved {len(results)} results to {OUTPUT_CSV}")
            last_checkpoint = current_time
        if min_hamming < BEST_HAMMING:
            search_range = int(search_range * 0.7)
            logger.info(f"Improved Hamming ({min_hamming}), shrinking range to ±{search_range}")
        else:
            search_range = int(search_range * 1.5)
            logger.info(f"No improvement, growing range to ±{search_range}")
        current_key += search_range
        iteration += 1
        elapsed = current_time - start_time
        logger.info(f"Iteration {iteration}: Tested {current_key}, Range ±{search_range}, Min Hamming {min_hamming}, Elapsed {elapsed:.2f}s")
        if any(r['Exact Match'] for r in results):
            print("COLLISION FOUND!")
            logger.info("Collision found! Stopping search.")
            write_csv(OUTPUT_CSV, results)
            write_csv(CHECKPOINT_FILE, results)
            break
    except Exception as e:
        logger.error(f"Error in iteration {iteration}: {e}")
        time.sleep(5)

logger.info("Search done. Check /content/ripemphantom_checkpoints")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 291.8 MB/s eta 0:00:00


Saving addresses.txt to addresses.txt


Iteration 639, Keys -9999999 to 10000001:  85%|████████▌ | 855/1001 [01:01<00:10, 13.91it/s]


KeyboardInterrupt: 

In [ ]:
# Install required packages:
# pip install pycryptodome base58 ecdsa numpy

import hashlib
from Crypto.Hash import RIPEMD160
import base58
import numpy as np
import csv
from ecdsa import SigningKey, SECP256k1

# Maximum SHA-256 passes to test
MAX_PASSES = 3
# Modulo value for tracking
MODULO = 256
# Modulo values to prioritize (from 0x006d816=22, 0x059ee05f=95, plus 0, 128, 160)
MODULO_TARGETS = [22, 95, 0, 128, 160]
# Key range (100,000 keys, neutral starting point)
START_KEY = 0x1
END_KEY = 0x186a0  # 100,000

# Read target addresses from addresses.txt
def load_target_addresses(file_path="addresses.txt"):
    try:
        with open(file_path, 'r') as f:
            addresses = [line.strip() for line in f if line.strip()]
        return addresses
    except FileNotFoundError:
        raise FileNotFoundError("addresses.txt not found. Please provide the file with 999 target addresses.")

# Decode address to get RIPEMD-160 hash
def get_ripemd160_from_address(address):
    try:
        decoded = base58.b58decode_check(address)
        return decoded[1:]  # Strip version byte
    except ValueError:
        return None

# Compute Hamming distance between two byte arrays
def hamming_distance(bytes1, bytes2):
    return np.sum([bin(b1 ^ b2).count('1') for b1, b2 in zip(bytes1, bytes2)])

# Compute chi-squared statistic for RIPEMD-160 hash
def chi_squared(hash_bytes):
    observed = np.array([b for b in hash_bytes])
    expected = 128  # Uniform expected value (256/2)
    chi2 = np.sum((observed - expected) ** 2 / expected)
    return chi2

# Generate Bitcoin address from private key
def private_key_to_address(priv_key_int, sha256_passes=1, compressed=True):
    priv_key_bytes = priv_key_int.to_bytes(32, 'big')
    sk = SigningKey.from_string(priv_key_bytes, curve=SECP256k1)
    pub_key = sk.get_verifying_key().to_string("compressed" if compressed else "uncompressed")
    h = pub_key
    for _ in range(sha256_passes):
        h = hashlib.sha256(h).digest()
    ripemd160 = RIPEMD160.new()
    ripemd160.update(h)
    hashed = ripemd160.digest()
    versioned = b'\x00' + hashed
    checksum = hashlib.sha256(hashlib.sha256(versioned).digest()).digest()[:4]
    address = base58.b58encode(versioned + checksum)
    return address.decode(), hashed

# Main function to test keys against multiple addresses
def main():
    output_file = "key_recovery_multi_sha_results.csv"
    with open(output_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Private_Key_Hex", "Target_Address", "SHA256_Passes", "Key_Format", "Hamming_Distance", "Chi_Squared", "Modulo_256"])

    # Load target addresses
    target_addresses = load_target_addresses()
    target_ripemd160s = {addr: get_ripemd160_from_address(addr) for addr in target_addresses}
    target_ripemd160s = {addr: ripemd for addr, ripemd in target_ripemd160s.items() if ripemd is not None}

    # Track global best Hamming distances per address, pass count, and key format
    global_best = {
        addr: {
            p: {
                fmt: {"distance": float('inf'), "key": None, "address": None, "chi2": None, "modulo": None}
                for fmt in ["compressed", "uncompressed"]
            }
            for p in range(1, MAX_PASSES + 1)
        }
        for addr in target_ripemd160s
    }

    # Test keys in range
    for key_int in range(START_KEY, END_KEY):
        modulo = key_int % MODULO
        log_priority = modulo in MODULO_TARGETS
        key_hex = hex(key_int)[2:].zfill(64)
        for passes in range(1, MAX_PASSES + 1):
            for compressed in [True, False]:
                key_format = "compressed" if compressed else "uncompressed"
                address, ripemd160 = private_key_to_address(key_int, passes, compressed)
                for target_addr, target_ripemd in target_ripemd160s.items():
                    distance = hamming_distance(ripemd160, target_ripemd)
                    chi2 = chi_squared(ripemd160)
                    # Update global best
                    if distance < global_best[target_addr][passes][key_format]["distance"]:
                        global_best[target_addr][passes][key_format] = {
                            "distance": distance, "key": key_hex, "address": address, "chi2": chi2, "modulo": modulo
                        }
                        if log_priority:
                            print(f"New best for {target_addr}, {passes} SHA-256 passes, {key_format}: Key {key_hex}, Address {address}, Hamming {distance}, Chi2 {chi2:.2f}, Modulo {modulo}")
                    # Stop if Hamming distance is 0
                    if distance == 0:
                        print(f"Exact match found! Key {key_hex}, Target {target_addr}, SHA-256 Passes {passes}, Format {key_format}, Chi2 {chi2:.2f}, Modulo {modulo}")
                        return
                    # Save all results to CSV
                    with open(output_file, 'a', newline='') as f:
                        writer = csv.writer(f)
                        writer.writerow([key_hex, target_addr, passes, key_format, distance, chi2, modulo])

    # Print final best results
    print("\nFinal Best Hamming Distances:")
    for target_addr in target_ripemd160s:
        for passes in range(1, MAX_PASSES + 1):
            for key_format in ["compressed", "uncompressed"]:
                data = global_best[target_addr][passes][key_format]
                if data["key"] is None:
                    print(f"{target_addr}, {passes} SHA-256 passes, {key_format}: No keys tested")
                else:
                    print(f"{target_addr}, {passes} SHA-256 passes, {key_format}: Best key is {data['key']} at {data['distance']} Hamming, Address {data['address']}, Chi2 {data['chi2']:.2f}, Modulo {data['modulo']}")

if __name__ == "__main__":
    main()

New best for 12tLs9c9RsALt4ockxa1hB4iTCTSmxj2me, 1 SHA-256 passes, compressed: Key 0000000000000000000000000000000000000000000000000000000000000016, Address 1GWqhXL1DGAEc6MVntAnmQe4GFnR9k2Ykd, Hamming 65, Chi2 739.27, Modulo 22
New best for 1DmXZ4UWKQ2AjwLAvEK8R6vmZe5jYoFUE6, 1 SHA-256 passes, compressed: Key 0000000000000000000000000000000000000000000000000000000000000016, Address 1GWqhXL1DGAEc6MVntAnmQe4GFnR9k2Ykd, Hamming 64, Chi2 739.27, Modulo 22
New best for 194RLDTv6rjkDu9kX99mTBuK6KunAWRRkk, 1 SHA-256 passes, compressed: Key 0000000000000000000000000000000000000000000000000000000000000016, Address 1GWqhXL1DGAEc6MVntAnmQe4GFnR9k2Ykd, Hamming 73, Chi2 739.27, Modulo 22
New best for 1KmaiBSWEwmLcmq1TZzEJqwvmchS6t2n3E, 1 SHA-256 passes, compressed: Key 0000000000000000000000000000000000000000000000000000000000000016, Address 1GWqhXL1DGAEc6MVntAnmQe4GFnR9k2Ykd, Hamming 72, Chi2 739.27, Modulo 22
New best for 1H1ECjR96iAiTpiVaooQwcV9JHxf57vg2m, 1 SHA-256 passes, compressed: Key 00000

In [ ]:
pip install base58 pycryptodome ecdsa torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 92.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [ ]:
import random
import hashlib
import base58
import torch
import torch.nn as nn
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160

# Global tracking for best Hamming distances and keys
best_hamming = {}
best_keys = {}

# Configuration
USE_NN = False  # Set to True if you confirm structured keys
MAX_K = 2**61  # Keyspace limit from ripphantom_t5.txt

# Neural Network for Hamming Distance Prediction (optional)
class HashDistanceNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.model(x)

def privkey_to_hash160(k: int, compressed: bool = True) -> bytes:
    """Generate RIPEMD160(SHA256(PubKey)) from a private key integer."""
    try:
        if not (1 <= k < SECP256k1.order):
            return None
        sk = SigningKey.from_secret_exponent(k, curve=SECP256k1)
        vk = sk.get_verifying_key()
        prefix = b'\x02' if vk.to_string()[32] % 2 == 0 else b'\x03'
        pubkey = prefix + vk.to_string()[:32] if compressed else b'\x04' + vk.to_string()
        sha = hashlib.sha256(pubkey).digest()
        ripemd = RIPEMD160.new()
        ripemd.update(sha)
        return ripemd.digest()
    except Exception:
        return None

def hamming_distance(hash1: bytes, hash2: bytes) -> int:
    """Calculate Hamming distance between two hash160 values."""
    distance = 0
    for b1, b2 in zip(hash1, hash2):
        xor = b1 ^ b2
        distance += bin(xor).count('1')
    return distance

def key_to_vector(k: int) -> torch.Tensor:
    """Convert private key to 256-bit binary tensor for NN."""
    return torch.tensor([(k >> i) & 1 for i in range(255, -1, -1)], dtype=torch.float32)

def load_addresses():
    """Load addresses from thekeymaster.csv."""
    addresses = []
    with open('thekeymaster.csv', 'r') as f:
        lines = f.readlines()
        for line in lines[1:]:  # Skip header
            addr = line.split(',')[0].strip()
            addresses.append(addr)

    targets = []
    for addr in addresses:
        try:
            decoded = base58.b58decode(addr)
            h160 = decoded[1:-4]
            if len(h160) == 20:
                targets.append((addr, h160))
        except:
            continue
    return targets

def mutate_key(k: int, nn_model=None, targets=None) -> int:
    """Mutate key, optionally using NN to guide mutations."""
    new_k = k
    if USE_NN and nn_model and targets:
        try:
            with torch.no_grad():
                k_tensor = key_to_vector(new_k).unsqueeze(0)
                predicted_hamming = nn_model(k_tensor).item()
                # If predicted Hamming is high, try a larger mutation
                if predicted_hamming > 50:
                    bit = random.randint(0, 60)
                    new_k ^= (1 << bit)
                else:
                    # Small mutation for fine-tuning
                    delta = random.randint(-100, 100)
                    new_k += delta
        except:
            # Fallback to random mutation if NN fails
            if random.random() < 0.5:
                bit = random.randint(0, 60)
                new_k ^= (1 << bit)
            else:
                delta = random.randint(-1000, 1000)
                new_k += delta
    else:
        # Simple mutation: bit flip or small increment
        if random.random() < 0.5:
            bit = random.randint(0, 60)
            new_k ^= (1 << bit)
        else:
            delta = random.randint(-1000, 1000)
            new_k += delta

    # Ensure key is within valid range
    if new_k < 1:
        new_k = 1
    if new_k >= MAX_K:
        new_k = random.randint(1, MAX_K - 1)
    return new_k

def main():
    global best_hamming, best_keys

    # Load addresses
    targets = load_addresses()
    print(f"Loaded {len(targets)} valid addresses.")

    # Initialize best Hamming distances
    for addr, _ in targets:
        best_hamming[addr] = 160
        best_keys[addr] = None

    # Initialize NN (if enabled)
    nn_model = None
    if USE_NN:
        nn_model = HashDistanceNet()
        nn_model.eval()
        print("Neural network initialized (note: untrained, effectiveness limited).")

    # Start with a random key
    current_key = random.randint(1, MAX_K - 1)
    attempts = 0

    while True:
        attempts += 1
        h160 = privkey_to_hash160(current_key)
        if not h160:
            current_key = random.randint(1, MAX_K - 1)
            continue

        for addr, target_h160 in targets:
            dist = hamming_distance(h160, target_h160)
            if dist == 0:
                print(f"\nWINNER! Found private key for {addr}")
                print(f"Private Key (hex): {hex(current_key)[2:].zfill(64)}")
                print(f"Attempts: {attempts}")
                return

            if dist < best_hamming[addr]:
                best_hamming[addr] = dist
                best_keys[addr] = current_key
                print(f"\nNew global best for {addr}:")
                print(f"Hamming Distance: {dist} bits")
                print(f"Private Key (hex): {hex(current_key)[2:].zfill(64)}")
                print(f"Attempts: {attempts}")

        # Mutate the current key
        current_key = mutate_key(current_key, nn_model, targets)

        if attempts % 100000 == 0:
            print(f"Attempts: {attempts}, still searching...")

if __name__ == "__main__":
    main()

Streaming output truncated to the last 5000 lines.

New global best for 1KbrSKrT3GeEruTuuYYUSQ35JwKbrAWJYm:
Hamming Distance: 49 bits
Private Key (hex): 0000000000000000000000000000000000000000000000001e2c41642a395c9d
Attempts: 609530

New global best for 1No2jiAeoHaKNGwa4S8P37D8Lo5WmoNENy:
Hamming Distance: 47 bits
Private Key (hex): 0000000000000000000000000000000000000000000000001423d38ab658345c
Attempts: 609757

New global best for 19ZiqyuBssnK1n31akRu8FUAEg8b4wiG9D:
Hamming Distance: 50 bits
Private Key (hex): 0000000000000000000000000000000000000000000000000bbb6b70d306cb9a
Attempts: 611825

New global best for 1Pg9WgAVUTJzcpVrBwtWWD3P3bVtvfEZhZ:
Hamming Distance: 50 bits
Private Key (hex): 0000000000000000000000000000000000000000000000001de4af0438792ab4
Attempts: 612315

New global best for 19FySy3ch3PjPHs4tGn71iKY8ThPi86HZX:
Hamming Distance: 48 bits
Private Key (hex): 0000000000000000000000000000000000000000000000000de48f2422612af1
Attempts: 612336

New global best for 1Fk5fWfQ

KeyboardInterrupt: 

In [1]:
import os
import random
import hashlib
import base58
import torch
import torch.nn as nn
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
import pickle
from multiprocessing import Process, cpu_count, Manager
from collections import defaultdict, deque

# Configuration
USE_NN = False
MAX_K = 2**256
CHECKPOINT_FILE = "checkpoint.pkl"
SOLVED_FILE = "solved.txt"
STRATEGY_WEIGHTS = {'bit_flip': 0.4, 'delta': 0.3, 'range_shift': 0.3}

# Neural Network (Placeholder, disabled)
class HashDistanceNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, 128), nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.model(x)

# Key-to-Hash160
def privkey_to_hash160(k: int, compressed: bool = True) -> bytes:
    try:
        if not (1 <= k < SECP256k1.order):
            return None
        sk = SigningKey.from_secret_exponent(k, curve=SECP256k1)
        vk = sk.get_verifying_key()
        pubkey = (b'\x02' if vk.to_string()[32] % 2 == 0 else b'\x03') + vk.to_string()[:32] if compressed else b'\x04' + vk.to_string()
        sha = hashlib.sha256(pubkey).digest()
        ripemd = RIPEMD160.new()
        ripemd.update(sha)
        return ripemd.digest()
    except:
        return None

# Hamming Distance
def hamming_distance_bits(hash1: bytes, hash2: bytes) -> tuple[int, list[int]]:
    distance = 0
    flips = []
    for i in range(len(hash1)):
        xor = hash1[i] ^ hash2[i]
        for j in range(8):
            if xor & (1 << j):
                bit_index = (i * 8) + (7 - j)
                flips.append(bit_index)
                distance += 1
    return distance, flips

# Load Addresses
def load_addresses():
    try:
        with open('addresses.txt', 'r') as f:
            addresses = [line.strip() for line in f if line.strip()]
        targets = []
        for addr in addresses:
            try:
                decoded = base58.b58decode(addr)
                h160 = decoded[1:-4]
                if len(h160) == 20:
                    targets.append((addr, h160))
            except:
                continue
        print(f"Loaded {len(targets)} valid addresses.")
        return targets
    except FileNotFoundError:
        print("Error: addresses.txt not found. Exiting.")
        exit(1)

# Checkpointing and Solved Output
def save_checkpoint(found_addresses: dict):
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(found_addresses, f)
    with open(SOLVED_FILE, 'a') as f:
        for addr, (key, compressed) in found_addresses.items():
            f.write(f"Address={addr}, Private Key={hex(key)}, Compressed={compressed}\n")

def load_checkpoint():
    found = {}
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'rb') as f:
            found = pickle.load(f)
        print(f"Loaded {len(found)} addresses from checkpoint.")
    return found

# Worker Function
def worker_task(args):
    tasks, shared_state = args
    for seed_k, target_addr, target_h160 in tasks:
        k = seed_k
        best_k = k
        best_ham = 160
        best_compressed = True
        iterations = 0
        update_count = 0
        local_bit_bias = defaultdict(float)
        local_mutation_bank = deque(maxlen=1000)
        local_weights = STRATEGY_WEIGHTS.copy()

        while True:
            iterations += 1
            new_best = False
            for compressed in [True, False]:
                h160 = privkey_to_hash160(k, compressed)
                if not h160:
                    continue
                ham, flips = hamming_distance_bits(h160, target_h160)
                if ham == 0:
                    print(f"RECOVERED: Address={target_addr}, Private Key={hex(k)}, Compressed={compressed}")
                    with shared_state['lock']:
                        shared_state['found'][target_addr] = (k, compressed)
                        save_checkpoint(shared_state['found'])
                    return
                if ham < best_ham:
                    best_k = k
                    best_ham = ham
                    best_compressed = compressed
                    new_best = True
                    if iterations > 100 and ham < shared_state['min_hamming'][target_addr] - 5:
                        with shared_state['lock']:
                            shared_state['min_hamming'][target_addr] = ham
                            shared_state['best_candidates'][target_addr] = (best_k, ham, best_compressed)
                        update_count += 1
                        print(f"Significant best for {target_addr}: {ham} bits off, Key={hex(k)}, Compressed={compressed}, Updates={update_count}")

            # Local mutation
            strategy = random.choices(
                list(local_weights.keys()),
                list(local_weights.values())
            )[0]
            new_k = k
            if strategy == 'bit_flip' and local_bit_bias:
                top_bits = sorted(local_bit_bias.items(), key=lambda x: -x[1])[:10]
                if top_bits:
                    bit = random.choices([b[0] for b in top_bits], [b[1] for b in top_bits])[0]
                    new_k ^= (1 << bit)
                else:
                    bit = random.randint(0, 255)
                    new_k ^= (1 << bit)
            elif strategy == 'delta' and local_mutation_bank:
                delta = random.choice(local_mutation_bank)['delta_k']
                new_k ^= delta
            else:
                new_k = random.randint(1, MAX_K - 1)

            if new_k < 1 or new_k >= MAX_K:
                new_k = random.randint(1, MAX_K - 1)

            h160 = privkey_to_hash160(new_k, compressed=True) or privkey_to_hash160(new_k, compressed=False)
            if not h160:
                k = new_k
                continue

            ham, flips = hamming_distance_bits(h160, target_h160)
            delta_hamming = prev_hamming = ham
            if delta_hamming > 0:
                for bit in flips:
                    local_bit_bias[bit] += delta_hamming * 0.2
                local_mutation_bank.append({
                    'delta_hamming': delta_hamming,
                    'flips': flips,
                    'delta_k': k ^ new_k
                })
                local_weights[strategy] *= (1.2 if delta_hamming > 0 else 0.8)
                total = sum(local_weights.values())
                for key in local_weights:
                    local_weights[key] /= total
            k = new_k

            if iterations % 1000 == 0 and best_ham > 50:
                k = random.randint(1, MAX_K - 1)
                best_ham = 160
                print(f"Worker for {target_addr} reset to explore, Iterations={iterations}, Updates={update_count}, Best Hamming={best_ham}")
            if iterations % 500 == 0:
                print(f"Worker for {target_addr} active, Iterations={iterations}, Best Hamming={best_ham}")

# Main Function
def main():
    targets = load_addresses()
    manager = Manager()
    shared_state = manager.dict({
        'lock': manager.Lock(),
        'found': manager.dict(),
        'min_hamming': manager.dict({addr: 160 for addr, _ in targets}),
        'best_candidates': manager.dict()
    })
    found_addresses = load_checkpoint()
    remaining_targets = [(random.randint(1, MAX_K - 1), addr, h160) for addr, h160 in targets if addr not in found_addresses]

    if not remaining_targets:
        print("All addresses recovered. Exiting.")
        return

    print(f"Targeting {len(remaining_targets)} remaining addresses.")

    num_workers = min(cpu_count(), 8)
    chunk_size = max(1, len(remaining_targets) // num_workers)
    tasks = [(remaining_targets[i:i + chunk_size], shared_state) for i in range(0, len(remaining_targets), chunk_size)]

    processes = []
    try:
        for task in tasks:
            p = Process(target=worker_task, args=(task,))
            processes.append(p)
            p.start()
        for p in processes:
            p.join()
    except KeyboardInterrupt:
        print("Interrupted. Saving checkpoint.")
        with shared_state['lock']:
            save_checkpoint(shared_state['found'])
        for p in processes:
            p.terminate()
    finally:
        for p in processes:
            p.close()

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'base58'